# Общие функции

Здесь только код, который повторяется в нескольких экспериментах: загрузка, нормализация,
метрика, поиск и признаки. Исследовательские решения и выводы остаются в 00--10.
Каждый основной ноутбук запускает этот файл сам через `%run`; вручную запускать его первым не нужно.
Функции не обучают модели и не читают данные при подключении.

## Загрузка и нормализация

In [ ]:
import hashlib

import json

import os

import re

import unicodedata

from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent

QUERY_COLS = [
    'search_query',
    'search_location_id',
    'search_is_delivery_search',
    'search_infm_params_text',
    'search_category',
]


In [ ]:
def normalize(value):
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', str(value)).lower().replace('ё', 'е')
    return ' '.join(re.findall(r'[\w]+', text))


In [ ]:
def data_dir():
    path = Path(os.environ.get('AVITO_DATA_DIR', ROOT / 'dataset'))
    if not (path / 'train.parquet').exists():
        raise FileNotFoundError(
            f'Положите три Parquet в {path} или задайте AVITO_DATA_DIR'
        )
    return path


In [ ]:
def load_raw():
    return tuple(
        pd.read_parquet(data_dir() / f'{name}.parquet')
        for name in ['train', 'benchmark_queries', 'benchmark_items']
    )


In [ ]:
def query_keys(frame):
    """Полный контекст сохраняет различие фильтров; cold split группируется по тексту."""
    values = frame[QUERY_COLS].copy()
    for col in QUERY_COLS:
        values[col] = (
            values[col].map(normalize)
            if col in ['search_query', 'search_infm_params_text']
            else values[col].astype('string').fillna('<NA>')
        )
    return values.apply(
        lambda row: json.dumps(row.tolist(), ensure_ascii=False, separators=(',', ':')),
        axis=1,
    )


In [ ]:
def prepare_queries(frame):
    result = frame[QUERY_COLS + (['query_id'] if 'query_id' in frame else [])].copy()
    result['text_key'] = result.search_query.map(normalize)
    result['context_key'] = query_keys(result)
    if 'query_id' not in result:
        result['query_id'] = result.context_key.map(
            lambda s: hashlib.sha256(s.encode()).hexdigest()[:24]
        )
    result['query_text'] = (
        result.text_key + ' ' + result.search_infm_params_text.map(normalize)
    )
    return result


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
def write_json(value, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8'
    )


## Метрика и формат ответа

In [ ]:
import csv

import re

from pathlib import Path

import numpy as np

import pandas as pd


In [ ]:
def recall_table(predictions, truth, ks=(1, 10, 20, 50, 100)):
    rows = []
    for query_id, relevant in truth.items():
        relevant = set(relevant)
        if not relevant:
            raise ValueError(f'Пустая разметка: {query_id}')
        ranked = predictions.get(query_id, [])
        if len(ranked) != len(set(ranked)):
            raise ValueError(f'Повторы в выдаче: {query_id}')
        rows.append(
            {
                'query_id': query_id,
                'n_relevant': len(relevant),
                **{
                    f'recall@{k}': len(set(ranked[:k]) & relevant) / len(relevant)
                    for k in ks
                },
            }
        )
    if not rows:
        raise ValueError('Нет запросов для оценки')
    return pd.DataFrame(rows)


In [ ]:
def bootstrap_ci(values, seed=143, repeats=2000):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = [
        rng.choice(values, len(values), replace=True).mean() for _ in range(repeats)
    ]
    return np.quantile(means, [0.025, 0.975]).tolist()


In [ ]:
def validate_submission(path, query_ids, item_ids):
    expected = list(query_ids)
    if len(set(expected)) != len(expected) or any(
        not isinstance(q, str) or len(q) != 16 for q in expected
    ):
        raise ValueError('Некорректные query_id в исходном benchmark')
    allowed = set(item_ids)
    if any(
        not isinstance(i, str) or re.fullmatch('[0-9a-f]{16}', i) is None
        for i in allowed
    ):
        raise ValueError('Некорректные item_id в исходном корпусе')
    with open(path, encoding='utf-8', newline='') as stream:
        reader = csv.reader(stream, strict=True)
        if next(reader, None) != ['query_id', 'answer']:
            raise ValueError('Ожидаются ровно query_id,answer в UTF-8 без BOM')
        rows = list(reader)
    seen = set()
    sizes = []
    for line, row in enumerate(rows, 2):
        if len(row) != 2:
            raise ValueError(f'Строка {line}: не две колонки')
        query_id, answer = row
        if query_id not in expected or query_id in seen:
            raise ValueError(f'Строка {line}: лишний или повторный query_id')
        seen.add(query_id)
        items = answer.split(' ') if answer else []
        if len(items) > 50 or len(set(items)) != len(items):
            raise ValueError(f'Строка {line}: >50 или повторные item_id')
        if any(
            re.fullmatch('[0-9a-f]{16}', item) is None or item not in allowed
            for item in items
        ):
            raise ValueError(f'Строка {line}: неправильный item_id или разделитель')
        sizes.append(len(items))
    if seen != set(expected):
        raise ValueError(f'Пропущено {len(set(expected) - seen)} query_id')
    return {
        'rows': len(rows),
        'min_candidates': min(sizes),
        'max_candidates': max(sizes),
        'valid': True,
    }


In [ ]:
def save_submission(predictions, queries, items, path):
    if set(predictions) != set(queries.query_id):
        raise ValueError('Набор query_id предсказаний не совпадает с benchmark')
    frame = pd.DataFrame(
        {
            'query_id': queries.query_id,
            'answer': [' '.join(predictions[q]) for q in queries.query_id],
        }
    )
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding='utf-8')
    return validate_submission(path, queries.query_id.tolist(), items.item_id.tolist())


## Признаки объявления и пары

In [ ]:
import re

import numpy as np

import pandas as pd


In [ ]:
def prepare_items(frame):
    items = frame.copy().reset_index(drop=True)
    for col in [
        'item_price',
        'item_latitude',
        'item_longitude',
        'item_rating',
        'item_rating_reviews_count',
    ]:
        items[col] = pd.to_numeric(items[col], errors='coerce').astype(float)
    for col, name in [
        ('item_title_raw', 'title_text'),
        ('item_infm_params_text', 'params_text'),
        ('item_description_raw', 'description_text'),
    ]:
        items[name] = items[col].map(normalize)
    # Lexical видит больше описания, dense укладывается в ограниченный token budget.
    items['lexical_text'] = (
        items.title_text
        + ' '
        + items.title_text
        + ' '
        + items.params_text.str[:800]
        + ' '
        + items.description_text.str[:2400]
    )
    items['dense_text'] = (
        items.title_text
        + '. '
        + items.params_text.str[:350]
        + '. '
        + items.description_text.str[:650]
    )
    return items


In [ ]:
def item_features(items):
    result = pd.DataFrame(index=items.index)
    for col in ['item_price', 'item_rating_reviews_count']:
        value = pd.to_numeric(items[col], errors='coerce')
        result[f'{col}_missing'] = value.isna().astype('int8')
        if col == 'item_price':
            result['item_price_unknown'] = (value.isna() | value.lt(0)).astype('int8')
            value = value.where(value.ge(0))
        result[f'{col}_log1p'] = np.log1p(value.clip(lower=0))
        result[f'{col}_zero'] = value.eq(0).astype('int8')
    result['rating'] = items.item_rating
    result['rating_missing'] = items.item_rating.isna().astype('int8')
    reviews = items.item_rating_reviews_count.fillna(0).clip(lower=0)
    # Prior считается по каталогу, а не по выбранным пользователями объявлениям.
    prior = items.item_rating.median()
    result['rating_shrunk'] = (
        reviews * items.item_rating.fillna(prior) + 20 * prior
    ) / (reviews + 20)
    for col in ['title_text', 'params_text', 'description_text']:
        result[col + '_loglen'] = np.log1p(items[col].str.len())
        result[col + '_empty'] = items[col].eq('').astype('int8')
    result['phone_hidden'] = items.item_is_phone_hidden.astype('int8')
    result['message_forbidden'] = items.item_is_message_forbidden.astype('int8')
    result['both_contacts_restricted'] = result.phone_hidden * result.message_forbidden
    price = items.item_price.where(items.item_price.ge(0))
    median = price.groupby(items.item_microcat_id).transform('median')
    result['price_vs_microcat_log'] = np.log1p(price) - np.log1p(median)
    result['latitude'] = items.item_latitude.where(items.item_latitude.between(-90, 90))
    result['longitude'] = items.item_longitude.where(
        items.item_longitude.between(-180, 180)
    )
    return result.astype('float32')


In [ ]:
def pair_features(query, positions, items, numeric, counts=None, centers=None):
    subset = items.iloc[positions]
    result = numeric.iloc[positions].reset_index(drop=True).copy()
    tokens = set(query.text_key.split())
    denom = max(1, len(tokens))
    for field in ['title_text', 'params_text', 'description_text']:
        result[field + '_coverage'] = [
            sum((' ' + t + ' ') in (' ' + x + ' ') for t in tokens) / denom
            for x in subset[field]
        ]
    result['query_tokens'] = len(tokens)
    result['query_chars'] = len(query.text_key)
    result['filter_chars'] = len(str(query.search_infm_params_text))
    result['same_location'] = (
        subset.item_location_id.eq(query.search_location_id).to_numpy().astype('int8')
    )
    result['same_category'] = (
        subset.item_category_id.eq(query.search_category).to_numpy().astype('int8')
    )
    result['delivery'] = int(query.search_is_delivery_search)
    result['item_pop_log'] = (
        np.log1p([counts.get(i, 0) for i in subset.item_id])
        if counts is not None
        else 0.0
    )
    result['distance_km_proxy'] = np.nan
    if centers is not None and query.search_location_id in centers.index:
        center = centers.loc[query.search_location_id]
        lat1, lon1 = np.radians([center.item_latitude, center.item_longitude])
        lat2 = np.radians(subset.item_latitude.to_numpy())
        lon2 = np.radians(subset.item_longitude.to_numpy())
        a = (
            np.sin((lat2 - lat1) / 2) ** 2
            + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
        )
        result['distance_km_proxy'] = (
            6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        )
    match = re.search(
        r'(\d(?:[.,]\d)?)\s*звезд', str(query.search_infm_params_text).lower()
    )
    result['rating_threshold_requested'] = int(match is not None)
    result['rating_margin'] = (
        subset.item_rating.to_numpy() - float(match[1].replace(',', '.'))
        if match
        else np.nan
    )
    return result.replace([np.inf, -np.inf], np.nan).astype('float32')


In [ ]:
def extra_features(frame, queries, items, idf, rank_columns=None):
    """Добавляю признаки без меток и без статистик отложенных выборов."""
    result = frame.copy()
    query_map = queries.set_index('query_id')
    titles = items.title_text.astype(object).to_numpy()
    params = items.params_text.astype(object).to_numpy()
    records = []

    for query_id, group in result.groupby('query_id', sort=False):
        query = query_map.loc[query_id]
        words = set(query.text_key.split())
        filters = set(normalize(query.search_infm_params_text).split())
        numbers = {word for word in words if any(char.isdigit() for char in word)}
        total_idf = sum(idf.get(word, 1.0) for word in words)

        for row in group.itertuples():
            title = titles[row.pos]
            title_words = set(title.split())
            param_words = set(params[row.pos].split())
            overlap = words & title_words
            records.append(
                {
                    'row': row.Index,
                    'title_jaccard': len(overlap) / max(1, len(words | title_words)),
                    'title_precision': len(overlap) / max(1, len(title_words)),
                    'title_idf_coverage': sum(idf.get(w, 1.0) for w in overlap)
                    / max(total_idf, 1.0),
                    'title_exact_phrase': int(
                        bool(query.text_key)
                        and (' ' + query.text_key + ' ') in (' ' + title + ' ')
                    ),
                    'title_extra_tokens': len(title_words - words),
                    'filter_param_coverage': len(filters & param_words)
                    / max(1, len(filters)),
                    'query_has_numbers': int(bool(numbers)),
                    'number_coverage': len(numbers & (title_words | param_words))
                    / max(1, len(numbers)),
                }
            )

    extra = pd.DataFrame(records).set_index('row').reindex(result.index)
    result = pd.concat([result, extra], axis=1)
    result['distance_log1p'] = np.log1p(result.distance_km_proxy.clip(lower=0))
    result['remote_distance'] = result.distance_log1p * (1 - result.same_location)
    result['dense_title_agreement'] = result.dense_cosine * result.title_text_coverage
    result['local_text_agreement'] = result.same_location * result.title_text_coverage
    result['rating_confidence'] = 1 - 1 / (
        1 + np.expm1(result.item_rating_reviews_count_log1p).clip(lower=0) / 20
    )

    if rank_columns is None:
        rank_columns = [col for col in frame if col.startswith('rank_')]
    # На инференсе использую тот же набор каналов, что при обучении.
    ranks = result.reindex(columns=rank_columns, fill_value=0)
    result['retriever_agreement'] = ranks.gt(0).sum(axis=1) / len(rank_columns)
    for name, tokens in [
        ('lexical', ['word', 'char', 'bm25']),
        ('semantic', ['dense', 'adapted']),
    ]:
        columns = [col for col in rank_columns if any(token in col for token in tokens)]
        result[f'{name}_best_rank'] = ranks[columns].max(axis=1)
    result['lexical_semantic_gap'] = (
        result.lexical_best_rank - result.semantic_best_rank
    )
    for name in ['dense_cosine', 'rrf', 'distance_km_proxy', 'item_price_log1p']:
        values = result.groupby('query_id')[name]
        result[f'{name}_pool_percentile'] = values.rank(
            pct=True, ascending=name != 'distance_km_proxy'
        )
    return result


## Текстовый поиск и история

In [ ]:
from collections import defaultdict

from pathlib import Path

import json

import time

import numpy as np

import pandas as pd

from scipy import sparse

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfTransformer,
    TfidfVectorizer,
)

from sklearn.naive_bayes import MultinomialNB


In [ ]:
def top_indices(scores, k=100, allowed=None):
    scores = np.asarray(scores).ravel()
    positions = np.arange(len(scores)) if allowed is None else np.asarray(allowed)
    if not len(positions):
        return np.empty(0, dtype=np.int32)
    values = scores[positions]
    valid = np.isfinite(values) & (values > 0)
    positions, values = positions[valid], values[valid]
    if len(positions) > k:
        # Все ties на границе проходят во второй этап: результат не зависит от argpartition.
        threshold = np.partition(values, -k)[-k]
        keep = values >= threshold
        positions, values = positions[keep], values[keep]
    order = np.lexsort((positions, -values))[:k]
    return positions[order].astype(np.int32)


In [ ]:
def sparse_top(row, k=100, allowed_mask=None):
    row = row.tocsr()
    idx, values = row.indices, row.data
    keep = values > 0
    if allowed_mask is not None:
        keep &= allowed_mask[idx]
    idx, values = idx[keep], values[keep]
    if len(idx) > k:
        threshold = np.partition(values, -k)[-k]
        keep = values >= threshold
        idx, values = idx[keep], values[keep]
    order = np.lexsort((idx, -values))[:k]
    return idx[order], values[order]


In [ ]:
class LexicalIndex:
    def __init__(self):
        self.word = CountVectorizer(
            min_df=2, max_features=160000, ngram_range=(1, 2), dtype=np.float32
        )
        self.title = CountVectorizer(
            min_df=2, max_features=100000, ngram_range=(1, 1), dtype=np.float32
        )
        self.char = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=3,
            max_features=180000,
            sublinear_tf=True,
            dtype=np.float32,
        )

    @staticmethod
    def bm25_matrix(counts, k1=1.2, b=0.75):
        lengths = np.asarray(counts.sum(axis=1)).ravel()
        df = np.asarray((counts > 0).sum(axis=0)).ravel()
        idf = np.log1p((counts.shape[0] - df + 0.5) / (df + 0.5))
        result = counts.copy().tocsr()
        norm = k1 * (1 - b + b * lengths / max(float(lengths.mean()), 1))
        result.data = (
            result.data
            * (k1 + 1)
            / (result.data + np.repeat(norm, np.diff(result.indptr)))
        )
        return result.multiply(idf.astype(np.float32)).tocsr()

    def fit(self, items):
        counts = self.word.fit_transform(items.lexical_text).tocsr()
        self.tfidf = TfidfTransformer(sublinear_tf=True)
        self.word_matrix = self.tfidf.fit_transform(counts).astype(np.float32)
        self.bm25 = self.bm25_matrix(counts)
        del counts
        title_counts = self.title.fit_transform(items.title_text).tocsr()
        self.overlap = title_counts.copy()
        self.overlap.data[:] = 1
        self.title_bm25 = self.bm25_matrix(title_counts)
        self.char_matrix = self.char.fit_transform(items.title_text).tocsr()
        return self

    def search(self, queries, items, k=120, methods=None, progress=True):
        methods = methods or ['overlap', 'word', 'char', 'bm25', 'bm25_title']
        qword = self.word.transform(queries.query_text)
        qtitle = self.title.transform(queries.text_key)
        qtitle.data[:] = 1
        qword_binary = qword.copy()
        qword_binary.data[:] = 1
        pairs = {
            'overlap': (qtitle, self.overlap),
            'word': (self.tfidf.transform(qword), self.word_matrix),
            'char': (self.char.transform(queries.text_key), self.char_matrix),
            'bm25': (qword_binary, self.bm25),
            'bm25_title': (qtitle, self.title_bm25),
        }
        locations = items.item_location_id.to_numpy()
        rows = []
        for method in methods:
            qm, im = pairs[method]
            start = time.monotonic()
            for batch in range(0, len(queries), 32):
                scores = (qm[batch : batch + 32] @ im.T).tocsr()
                for offset in range(scores.shape[0]):
                    q = queries.iloc[batch + offset]
                    row = scores.getrow(offset)
                    for suffix, mask in [
                        ('', None),
                        ('_local', locations == q.search_location_id),
                    ]:
                        pos, values = sparse_top(row, k, mask)
                        rows.extend(
                            (q.query_id, int(p), method + suffix, rank + 1, float(s))
                            for rank, (p, s) in enumerate(zip(pos, values))
                        )
            if progress:
                print(
                    f'{method}: {len(queries)} queries, {time.monotonic()-start:.1f}s',
                    flush=True,
                )
        return compact_candidates(
            pd.DataFrame(rows, columns=['query_id', 'pos', 'source', 'rank', 'score'])
        )


In [ ]:
class HistoryIndex:
    def fit(self, pairs, items, label_catalog=None):
        self.items = items[
            ['item_id', 'item_location_id', 'item_category_id', 'item_microcat_id']
        ].copy()
        self.position = dict(zip(items.item_id, items.index))
        self.pop = pairs.item_id.value_counts().to_dict()
        counts = np.array([self.pop.get(i, 0) for i in items.item_id], dtype=np.float32)
        self.pop_scores = counts
        self.context = self._hist(pairs, 'context_key')
        self.text = self._hist(pairs, 'text_key')
        unique = (
            pairs[pairs.text_key.isin(self.text)]
            .drop_duplicates('text_key')
            .sort_values('text_key')
        )
        self.hist_texts = unique.text_key.tolist()
        self.query_vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=1,
            max_features=100000,
            dtype=np.float32,
        )
        self.query_matrix = self.query_vectorizer.fit_transform(self.hist_texts)
        meta = (
            (items if label_catalog is None else label_catalog)
            .set_index('item_id')
            .item_microcat_id
        )
        train = pairs[pairs.item_id.isin(meta.index)].copy()
        # Naive Bayes предсказывает тип услуги, а не 500 тысяч классов item_id.
        self.nb_vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_features=80000, dtype=np.float32
        )
        x = self.nb_vectorizer.fit_transform(train.query_text)
        self.nb = MultinomialNB(alpha=0.2).fit(x, train.item_id.map(meta).astype(str))
        self.microcat_positions = {
            str(cat): np.asarray(idx, dtype=np.int32)
            for cat, idx in items.groupby('item_microcat_id').groups.items()
        }
        return self

    def _hist(self, pairs, key):
        counts = pairs.groupby([key, 'item_id']).size().rename('count').reset_index()
        out = defaultdict(list)
        for row in counts.itertuples(index=False):
            p = self.position.get(row.item_id)
            if p is not None:
                out[getattr(row, key)].append((p, float(row.count)))
        return {
            key: sorted(values, key=lambda x: (-x[1], x[0]))
            for key, values in out.items()
        }

    def search(self, queries, k=120):
        rows = []
        locations = self.items.item_location_id.to_numpy()
        categories = self.items.item_category_id.to_numpy()
        near = (
            self.query_vectorizer.transform(queries.text_key) @ self.query_matrix.T
        ).tocsr()
        nb_probs = self.nb.predict_proba(
            self.nb_vectorizer.transform(queries.query_text)
        )
        for j, q in enumerate(queries.itertuples(index=False)):
            local = np.flatnonzero(locations == q.search_location_id)
            category = np.flatnonzero(categories == q.search_category)
            channels = {
                'popularity': [
                    (int(p), self.pop_scores[p])
                    for p in top_indices(self.pop_scores, k)
                ],
                'popularity_local': [
                    (int(p), self.pop_scores[p])
                    for p in top_indices(self.pop_scores, k, local)
                ],
                'popularity_category': [
                    (int(p), self.pop_scores[p])
                    for p in top_indices(self.pop_scores, k, category)
                ],
                'history': self.text.get(q.text_key, [])[:k],
                'history_context': self.context.get(q.context_key, [])[:k],
                'history_local': [
                    (p, s)
                    for p, s in self.text.get(q.text_key, [])
                    if locations[p] == q.search_location_id
                ][:k],
            }
            npos, nsim = sparse_top(near.getrow(j), 8)
            similar = defaultdict(float)
            for p, s in zip(npos, nsim):
                # Exact history -- отдельный канал; не удваиваю ее этим источником.
                if self.hist_texts[p] == q.text_key or s < 0.2:
                    continue
                for item, count in self.text.get(self.hist_texts[p], [])[:200]:
                    similar[item] += float(s) ** 3 * np.log1p(count)
            channels['similar_history'] = sorted(
                similar.items(), key=lambda x: (-x[1], x[0])
            )[:k]
            channels['similar_history_local'] = sorted(
                (
                    (p, s)
                    for p, s in similar.items()
                    if locations[p] == q.search_location_id
                ),
                key=lambda x: (-x[1], x[0]),
            )[:k]
            cats = np.argsort(-nb_probs[j])[:3]
            nb_scores = {}
            for catpos in cats:
                pos = self.microcat_positions.get(
                    str(self.nb.classes_[catpos]), np.array([], dtype=int)
                )
                for p in top_indices(self.pop_scores + 1, k, pos):
                    nb_scores[int(p)] = float(nb_probs[j, catpos]) * float(
                        np.log1p(self.pop_scores[p]) + 1
                    )
            channels['naive_bayes'] = sorted(
                nb_scores.items(), key=lambda x: (-x[1], x[0])
            )[:k]
            channels['naive_bayes_local'] = []
            for catpos in cats:
                pos = self.microcat_positions.get(
                    str(self.nb.classes_[catpos]), np.array([], dtype=int)
                )
                pos = pos[locations[pos] == q.search_location_id]
                for p in top_indices(self.pop_scores + 1, k, pos):
                    channels['naive_bayes_local'].append(
                        (
                            int(p),
                            float(nb_probs[j, catpos])
                            * (np.log1p(self.pop_scores[p]) + 1),
                        )
                    )
            channels['naive_bayes_local'].sort(key=lambda x: (-x[1], x[0]))
            for source, values in channels.items():
                rows.extend(
                    (q.query_id, p, source, rank + 1, float(s))
                    for rank, (p, s) in enumerate(values[:k])
                )
        return compact_candidates(
            pd.DataFrame(rows, columns=['query_id', 'pos', 'source', 'rank', 'score'])
        )


In [ ]:
def rrf(candidates, weights=None, constant=30, top_k=100):
    frame = candidates.copy()
    weights = (
        weights if weights is not None else {s: 1.0 for s in frame.source.unique()}
    )
    frame['rrf'] = frame.source.map(weights).fillna(0) / (constant + frame['rank'])
    frame = (
        frame[frame.rrf.gt(0)].groupby(['query_id', 'pos'], as_index=False).rrf.sum()
    )
    return (
        frame.sort_values(['query_id', 'rrf', 'pos'], ascending=[True, False, True])
        .groupby('query_id', sort=False)
        .head(top_k)
    )


In [ ]:
def predictions_from_frame(frame, items, score=None, k=100):
    if score is not None:
        frame = frame.sort_values(
            ['query_id', score, 'pos'], ascending=[True, False, True]
        )
    frame = (
        frame.drop_duplicates(['query_id', 'pos'])
        .groupby('query_id', sort=False)
        .head(k)
    )
    ids = items.item_id.to_numpy()
    frame = frame.assign(item_id=ids[frame.pos.to_numpy(dtype=int)])
    return frame.groupby('query_id', sort=False).item_id.agg(list).to_dict()


In [ ]:
def dense_search(query_vectors, item_vectors, queries, items, k=120, batch_size=16):
    rows = []
    locations = items.item_location_id.to_numpy()
    for start in range(0, len(queries), batch_size):
        scores = (
            np.asarray(query_vectors[start : start + batch_size], dtype=np.float32)
            @ np.asarray(item_vectors, dtype=np.float32).T
        )
        for offset, values in enumerate(scores):
            q = queries.iloc[start + offset]
            for source, allowed in [
                ('dense', None),
                ('dense_local', np.flatnonzero(locations == q.search_location_id)),
            ]:
                positions = top_indices(values, k, allowed)
                rows.extend(
                    (q.query_id, int(p), source, rank + 1, float(values[p]))
                    for rank, p in enumerate(positions)
                )
    return compact_candidates(
        pd.DataFrame(rows, columns=['query_id', 'pos', 'source', 'rank', 'score'])
    )


In [ ]:
def compact_candidates(frame):
    return frame.astype(
        {
            'query_id': 'string',
            'pos': 'int32',
            'source': 'category',
            'rank': 'int16',
            'score': 'float32',
        }
    )


## Объединение кандидатов и матрица модели

In [ ]:
import numpy as np

import pandas as pd


In [ ]:
def candidate_features(
    candidates,
    queries,
    items,
    numeric,
    counts,
    weights,
    query_vectors=None,
    item_vectors=None,
    max_candidates=400,
    centers=None,
):
    pool = rrf(candidates, weights, constant=30, top_k=max_candidates)
    # Схема фиксируется по названиям, а не по случайно встретившимся в batch каналам.
    keys = pool[['query_id', 'pos']]
    selected = candidates.merge(keys, on=['query_id', 'pos'], how='inner')
    rank = selected.pivot_table(
        index=['query_id', 'pos'],
        columns='source',
        values='rank',
        aggfunc='min',
        observed=True,
    )
    rank = (1 / (30 + rank)).fillna(0).add_prefix('rank_')
    score = (
        selected.pivot_table(
            index=['query_id', 'pos'],
            columns='source',
            values='score',
            aggfunc='max',
            observed=True,
        )
        .fillna(0)
        .add_prefix('score_')
    )
    source_features = rank.join(score).reset_index()
    source_features['n_sources'] = (rank > 0).sum(axis=1).to_numpy()
    if centers is None:
        centers = items.groupby('item_location_id')[
            ['item_latitude', 'item_longitude']
        ].median()
    # Arrow chunked strings при многократном take могут копировать весь буфер.
    # Оставляю только нужные поля и один раз материализую строки для быстрых gather.
    pair_items = items[
        [
            'item_id',
            'title_text',
            'params_text',
            'description_text',
            'item_location_id',
            'item_category_id',
            'item_rating',
            'item_latitude',
            'item_longitude',
        ]
    ].copy()
    for col in ['item_id', 'title_text', 'params_text', 'description_text']:
        pair_items[col] = pair_items[col].astype(object)
    query_map = queries.set_index('query_id')
    frames = []
    for query_id, group in pool.groupby('query_id', sort=False):
        q = query_map.loc[query_id]
        positions = group.pos.to_numpy(dtype=int)
        features = pair_features(q, positions, pair_items, numeric, counts, centers)
        features['query_id'] = query_id
        features['pos'] = positions
        features['rrf'] = group.rrf.to_numpy()
        if query_vectors is not None and item_vectors is not None:
            features['dense_cosine'] = (
                np.asarray(item_vectors[positions]) @ query_vectors[query_id]
            )
        frames.append(features)
    result = pd.concat(frames, ignore_index=True).merge(
        source_features, on=['query_id', 'pos'], how='left'
    )
    return result


In [ ]:
def feature_columns(frame):
    return [
        c
        for c in frame
        if c not in ['query_id', 'pos', 'target', 'regime', 'split', 'item_id']
    ]


In [ ]:
def model_matrix(frame, columns):
    # Источник может не вернуть ни одного кандидата для batch: missing rank/score = 0.
    result = frame.reindex(columns=columns).copy()
    for col in columns:
        if col.startswith(('rank_', 'score_')) or col == 'n_sources':
            result[col] = result[col].fillna(0)
    return result.fillna(-1).astype('float32')
